# Shenzhen holdout recovery run — Colab

Clean Colab path for the next TB recovery experiment:
1. prepare Shenzhen holdout metadata
2. train DenseNet121 with partial unfreeze (`0.25`), LR `3e-5`, no class weighting, mild augmentation
3. analyze internal thresholds
4. evaluate on the true unseen Shenzhen holdout

This notebook only uses the repo's current supported CLI scripts and flags.

## 0) Runtime setup

Use a **GPU runtime** in Colab. This notebook assumes you will provide the repo's processed data so the clone has either `data/processed/merged_metadata.csv` or the legacy `data/merged_metadata.csv`, plus `data/processed/extracted/...`.

In [ ]:
%cd /content
!git clone <YOUR-REPO-URL> tb-triage-v2
%cd /content/tb-triage-v2
!pip install -r requirements-colab.txt

In [ ]:
import json
import shutil
from pathlib import Path

REPO_ROOT = Path('/content/tb-triage-v2')
RUN_DIR = REPO_ROOT / 'experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5'
EXPERIMENT_METADATA = REPO_ROOT / 'data/processed/source_holdout/shenzhen_experiment_metadata.csv'
HOLDOUT_METADATA = REPO_ROOT / 'data/processed/source_holdout/shenzhen_holdout_only.csv'

print('repo_root:', REPO_ROOT)
print('run_dir:', RUN_DIR)

## 1) Optional: mount Google Drive

If your prepared processed data lives in Drive, mount it first.

In [ ]:
# Uncomment if needed
# from google.colab import drive
# drive.mount('/content/drive')

## 2) Put the processed dataset in place

Expected minimum layout:
- `data/processed/merged_metadata.csv`
- `data/processed/extracted/...`

Update the source path below if you want to copy a prepared `processed/` directory from Drive.

In [ ]:
SOURCE_PROCESSED_DIR = Path('/content/drive/MyDrive/tb-triage-v2-data/processed')

if SOURCE_PROCESSED_DIR.exists():
    shutil.rmtree(REPO_ROOT / 'data' / 'processed', ignore_errors=True)
    shutil.copytree(SOURCE_PROCESSED_DIR, REPO_ROOT / 'data' / 'processed')
    print('Copied processed data from', SOURCE_PROCESSED_DIR)
else:
    print('SOURCE_PROCESSED_DIR does not exist yet. Update it or skip this cell if you copied data another way.')

In [ ]:
required_paths = [
    REPO_ROOT / 'data/processed/extracted',
]

for p in required_paths:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')

print('processed merged metadata exists:', (REPO_ROOT / 'data/processed/merged_metadata.csv').exists())
print('legacy merged metadata exists:', (REPO_ROOT / 'data/merged_metadata.csv').exists())

assert (REPO_ROOT / 'data/processed/extracted').exists(), 'Missing data/processed/extracted'
assert (REPO_ROOT / 'data/processed/merged_metadata.csv').exists() or (REPO_ROOT / 'data/merged_metadata.csv').exists(), 'Missing merged metadata CSV'

## 3) Prepare the Shenzhen holdout metadata

This writes the explicit seen-source experiment CSV plus the true unseen Shenzhen holdout CSV. The helper already falls back to the legacy `data/merged_metadata.csv` path when needed.

In [ ]:
!python scripts/colab_prepare_source_holdout.py \
  --repo-root /content/tb-triage-v2 \
  --metadata-csv data/processed/merged_metadata.csv \
  --holdout-source shenzhen

In [ ]:
import pandas as pd

experiment_df = pd.read_csv(EXPERIMENT_METADATA)
holdout_df = pd.read_csv(HOLDOUT_METADATA)

print('experiment rows:', len(experiment_df))
print('holdout rows:', len(holdout_df))
display(experiment_df['experiment_split'].value_counts(dropna=False).rename_axis('experiment_split').to_frame('rows'))
display(holdout_df['label_final'].value_counts(dropna=False).rename_axis('label_final').to_frame('rows'))

## 4) Train the recovery model

Exact current recovery recipe from the repo:
- architecture: `densenet121`
- `--trainable-fraction 0.25`
- `--learning-rate 3e-5`
- `--class-weight none`
- `--augmentation mild`
- `--epochs 20`
- Shenzhen held out as the true unseen source

In [ ]:
!python scripts/colab_train_baseline.py \
  --repo-root /content/tb-triage-v2 \
  --metadata-csv data/processed/source_holdout/shenzhen_experiment_metadata.csv \
  --output-dir experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5 \
  --architecture densenet121 \
  --epochs 20 \
  --batch-size 32 \
  --image-size 256 \
  --learning-rate 3e-5 \
  --trainable-fraction 0.25 \
  --class-weight none \
  --augmentation mild

In [ ]:
metrics_path = RUN_DIR / 'metrics.json'
history_path = RUN_DIR / 'history.json'
model_path = RUN_DIR / 'densenet121_baseline.keras'

for p in [metrics_path, history_path, model_path]:
    print(f'{p.name}:', 'OK' if p.exists() else 'MISSING')

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))

## 5) Analyze thresholds on the seen-source internal test split

This keeps the Shenzhen holdout untouched while you pick a candidate operating threshold from the seen-source side.

In [ ]:
!python scripts/colab_analyze_thresholds.py \
  --repo-root /content/tb-triage-v2 \
  --metadata-csv data/processed/source_holdout/shenzhen_experiment_metadata.csv \
  --run-dir experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5 \
  --architecture densenet121 \
  --target-recall 0.90

In [ ]:
threshold_dir = RUN_DIR / 'threshold_analysis'
threshold_metrics_path = threshold_dir / 'threshold_metrics.csv'
selected_threshold_path = threshold_dir / 'selected_threshold_for_target_recall.json'

threshold_df = pd.read_csv(threshold_metrics_path)
display(threshold_df)

selected_threshold = 0.40
if selected_threshold_path.exists():
    with open(selected_threshold_path) as f:
        selected_info = json.load(f)
    selected_threshold = float(selected_info['selected_threshold'])
    print('Selected threshold from target recall search:', selected_info)
else:
    print('No threshold met the target recall. Keeping 0.40 as the continuity fallback.')

print('Threshold to reuse for holdout eval:', selected_threshold)

## 6) Evaluate on the true unseen Shenzhen holdout

Run two external evaluations:
- threshold `0.40` for continuity with earlier comparisons
- the threshold selected from the seen-source threshold analysis

In [ ]:
!python scripts/colab_eval_external.py \
  --repo-root /content/tb-triage-v2 \
  --metadata-csv data/processed/source_holdout/shenzhen_holdout_only.csv \
  --run-dir experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5 \
  --architecture densenet121 \
  --threshold 0.40

In [ ]:
!python scripts/colab_eval_external.py \
  --repo-root /content/tb-triage-v2 \
  --metadata-csv data/processed/source_holdout/shenzhen_holdout_only.csv \
  --run-dir experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5 \
  --architecture densenet121 \
  --threshold {selected_threshold} \
  --output-dir experiments/source_holdout/shenzhen_densenet121_partial_unfreeze_lr3e5/external_eval/shenzhen_holdout_selected_threshold

In [ ]:
external_default_dir = RUN_DIR / 'external_eval' / 'shenzhen_holdout_only'
external_selected_dir = RUN_DIR / 'external_eval' / 'shenzhen_holdout_selected_threshold'

for name, eval_dir in [('default_0.40', external_default_dir), ('selected_threshold', external_selected_dir)]:
    metrics_json = eval_dir / 'metrics.json'
    confusion_csv = eval_dir / 'confusion_at_threshold.csv'
    print('\n===', name, '===')
    print('dir:', eval_dir)
    print('metrics exists:', metrics_json.exists())
    print('confusion exists:', confusion_csv.exists())
    if metrics_json.exists():
        with open(metrics_json) as f:
            print(json.dumps(json.load(f), indent=2))
    if confusion_csv.exists():
        display(pd.read_csv(confusion_csv))

## 7) Bundle the key artifacts

This collects the main outputs from the holdout prep, training, threshold analysis, and both holdout evaluations into one zip for download.

In [ ]:
bundle_dir = REPO_ROOT / 'artifacts' / 'shenzhen_densenet121_recovery_bundle'
shutil.rmtree(bundle_dir, ignore_errors=True)
bundle_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    EXPERIMENT_METADATA,
    HOLDOUT_METADATA,
    RUN_DIR / 'metrics.json',
    RUN_DIR / 'history.json',
    RUN_DIR / 'densenet121_baseline.keras',
    RUN_DIR / 'threshold_analysis' / 'threshold_metrics.csv',
    RUN_DIR / 'threshold_analysis' / 'test_predictions.csv',
    RUN_DIR / 'threshold_analysis' / 'selected_threshold_for_target_recall.json',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_only' / 'metrics.json',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_only' / 'predictions.csv',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_only' / 'confusion_at_threshold.csv',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_only' / 'threshold_metrics.csv',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_selected_threshold' / 'metrics.json',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_selected_threshold' / 'predictions.csv',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_selected_threshold' / 'confusion_at_threshold.csv',
    RUN_DIR / 'external_eval' / 'shenzhen_holdout_selected_threshold' / 'threshold_metrics.csv',
]

copied = []
for src in files_to_copy:
    if src.exists():
        relative = src.relative_to(REPO_ROOT)
        dst = bundle_dir / relative
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        copied.append(dst)

print('Copied artifact files:')
for p in copied:
    print('-', p.relative_to(bundle_dir))

zip_base = str(bundle_dir)
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=bundle_dir)
print('Created zip:', zip_path)

In [ ]:
from google.colab import files
files.download(str(REPO_ROOT / 'artifacts' / 'shenzhen_densenet121_recovery_bundle.zip'))